# ML-09: Validation and Research Claim Audit

**Task:** audit two findings from FlyRank's own research paper ("The State of AI-Driven SEO,
March 2026"), then turn the same lens on my own Week 5 model. Page citations below are taken
directly from the shipped PDF, not the session video's summary of it.


## 1) Two Paper Findings + Methodology Questions

### Finding 1 — Health Score "predicted" by Random Forest (pages 5 and 27)

**Page 5** defines Health Score by hand: Impressions (30 pts) + Position (30 pts) + CTR (20 pts)
+ Scroll Depth (20 pts) — a formula, not a discovered outcome.

**Page 27** reports Random Forest feature importance for predicting that same Health Score:
Average Position 43%, Impressions 32%, Scroll Depth 15%, CTR 8% — summing to 98%. Every feature
not in the hand-built formula sits at 0%.

**Methodology question:** *Where does the label come from?* The label (Health Score) is
definitionally constructed from three of the four "top predictors" the model surfaces. A model
finding that position/impressions/CTR/scroll-depth predict a score built out of
position/impressions/CTR/scroll-depth is close to circular, not a discovery of what actually
drives page health beyond the formula itself. To the paper's credit, it says this directly: "the
target itself is partly constructed from some of these inputs, so importance is descriptive
rather than causal" — the honest caveat is present, but "partly constructed" still slightly
undersells a 98-of-100 overlap between formula ingredients and top features.

### Finding 2 — 71% holdout accuracy predicting growth vs. decline (pages 29 and 36)

**Page 29:** "Logistic regression (71% holdout accuracy) describing which sampled features
separate growing from declining pages," evaluated on an 80/20 split.

**Page 36 (Methodology):** "Random Forest (80/20 split), Logistic Regression (80/20 split)... ML
pages remain exploratory." That is the entire description of the split — no mention of whether
rows were held out randomly, grouped by client, or separated by time.

**Methodology question:** *Does the validation design support the claim?* An 80/20 split says how
many rows were held out, not which things were kept apart. If pages from the same client appear
in both the 80% training portion and the 20% holdout, a model can partially learn that specific
client's typical growth pattern rather than a generalizable one — the exact failure mode this
internship's own Week 5 session demonstrated (a model's apparent accuracy collapsing once the
split changed from random rows to held-out clients). The paper's methodology section gives no
evidence this was checked. 71% accuracy is a real number, but without knowing what was actually
held out, it isn't clear whether it would survive being tested against unseen clients or a
forward-in-time deployment.


## 2) My Model Under an Honest Split — Before/After

Week 5 already tested one honest split (grouped by `client_hash_id`, within March). This week
adds a second, different honest test the paper's own gap points at directly: **time-aware
validation** — train on an earlier month, test on a later one, the way the model would actually
be deployed (trained once, used going forward, not retrained every day). If Week 5's Random
Forest only works within the month it was trained on, that's a real limitation the client-grouped
test alone wouldn't catch.

**Before:** train and test both within March (Week 5's baseline framing, random split — the
naive, too-easy version).
**After:** train on February, test on March — deployment-realistic, nothing from March seen
during training.


In [1]:
import duckdb
from google.colab import userdata

con = duckdb.connect()
con.sql("INSTALL httpfs;")
con.sql("LOAD httpfs;")
con.sql(f"""
CREATE SECRET hf_secret (
TYPE huggingface,
TOKEN '{userdata.get("HF_TOKEN")}'
);
""")

print("Setup complete.")

Setup complete.


In [2]:
FEB_PATH = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/*.parquet"
MARCH_PATH = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"
CONTENT_PATH = "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet"

def build_feature_frame(path):
    query = f"""
    WITH agg AS (
    SELECT
    client_hash_id, content_hash_id,
    SUM(gsc_clicks) AS clicks,
    SUM(gsc_impressions) AS impressions,
    SUM(gsc_avg_position * gsc_impressions) / NULLIF(SUM(gsc_impressions), 0) AS avg_position
    FROM read_parquet('{path}')
    GROUP BY client_hash_id, content_hash_id
    HAVING SUM(gsc_impressions) >= 100
    )
    SELECT
    a.client_hash_id, a.content_hash_id,
    a.avg_position, a.impressions,
    d.main_intent, d.search_volume, d.category_count,
    a.clicks * 1.0 / NULLIF(a.impressions, 0) AS actual_ctr
    FROM agg a
    JOIN read_parquet('{CONTENT_PATH}') d ON a.content_hash_id = d.content_hash_id
    """
    df = con.sql(query).df().dropna(subset=["actual_ctr", "avg_position", "main_intent",
                                             "search_volume", "category_count", "impressions"])
    numeric_cols = ["avg_position", "impressions", "search_volume", "category_count"]
    df[numeric_cols] = df[numeric_cols].astype("float64")
    return df

df_feb = build_feature_frame(FEB_PATH)
df_march = build_feature_frame(MARCH_PATH)
print("February rows:", df_feb.shape, " March rows:", df_march.shape)

February rows: (78396, 8)  March rows: (99197, 8)


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error

numeric_features = ["avg_position", "impressions", "search_volume", "category_count"]
categorical_features = ["main_intent"]

def build_pipeline():
    pre = ColumnTransformer([
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ])
    return Pipeline([
        ("pre", pre),
        ("rf", RandomForestRegressor(n_estimators=300, max_depth=8, random_state=42, n_jobs=-1)),
    ])

# BEFORE: naive within-March random split (Week 5's easy version)
X_march = df_march[numeric_features + categorical_features]
y_march = df_march["actual_ctr"]
X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(X_march, y_march, test_size=0.25, random_state=42)
model_before = build_pipeline()
model_before.fit(X_train_b, y_train_b)
mae_before = mean_absolute_error(y_test_b, model_before.predict(X_test_b))

# AFTER: time-aware — train on February, test on all of March (nothing from March seen in training)
X_feb = df_feb[numeric_features + categorical_features]
y_feb = df_feb["actual_ctr"]
model_after = build_pipeline()
model_after.fit(X_feb, y_feb)
mae_after = mean_absolute_error(y_march, model_after.predict(X_march))

print(f"BEFORE (within-March, random split):     MAE = {mae_before:.5f}")
print(f"AFTER  (trained Feb, tested on March):    MAE = {mae_after:.5f}")
print(f"\nGap: {mae_after - mae_before:.5f} ({(mae_after / mae_before - 1) * 100:+.1f}%)")


BEFORE (within-March, random split):   MAE = 0.00243
AFTER  (trained Feb, tested on March):  MAE = 0.00243
Gap: -0.00000 (-0.0%)


**Real result:** BEFORE (within-March, random split): MAE = 0.00243. AFTER (trained on
February, tested on March): MAE = 0.00243. Gap: essentially zero (-0.0%).

**This is a genuinely reassuring finding, not a null result.** Training on a month the model
never saw performed exactly as well as training and testing within the same month. This makes
sense given Week 5's permutation importance: nearly all of the model's signal traces to
`avg_position` alone, and a position-to-CTR relationship is a stable pattern — a page ranking
around position 5 tends to earn roughly similar CTR whether it's February or March — unlike a
client-specific quirk, which is what the Week 5 grouped-by-client test was built to catch. This
time-aware result is a *stronger* claim than Week 5's grouped-split result alone could support:
the model isn't just avoiding client-level memorization, it also isn't relying on anything
specific to March's particular conditions. Reported at its real, unremarkable-looking size — a
near-zero gap is the finding, not a sign the test didn't do anything.


## 3) Leakage Audit

Going through each of the five features used since Week 5, checking whether it could leak
information from the target or from outside the decision-moment window:

| Feature | Audit |
|---|---|
| `avg_position` | Computed from the same March GSC data as the target's numerator (clicks) and denominator (impressions), but is not itself derived from CTR — position and CTR are two separate signals GSC reports. No leakage. |
| `impressions` | Used both as a feature AND as the denominator of `actual_ctr`. This is a real double-use worth flagging explicitly, though not classic leakage — impressions is a legitimate predictor (visibility/authority), and Week 5's permutation importance already showed it carries close to zero signal (-0.0013), which is itself evidence against it functioning as a disguised copy of the target. Flagged and monitored, not treated as clean by default. |
| `main_intent` | A static content-level attribute (from `dim_content`), set independently of any month's performance. No leakage. |
| `search_volume` | A keyword-market fact, independent of how this specific content item performs. No leakage. |
| `category_count` | A static content-level attribute, same as `main_intent`. No leakage. |

**No feature here is a disguised copy of the label** the way the Week 3 trap (`ctr_tier`, a binned
version of the target) was. The one flagged item (`impressions`) is a legitimate predictor with an
honest double-use, not a leak — but it's the kind of thing worth re-checking if this model's
result ever looks suspiciously good, the same discipline used to catch the Week 3 trap.


## 4) Claim Rewrite

**Original (Week 5/8) claim, as first stated:** "Random Forest beats the baseline by 3.2%."

That sentence implies a settled, general result. It is neither — it came from a single grouped
split, and permutation importance showed most of the gain traces to one feature (`avg_position`)
being modeled more flexibly than a five-bucket average, not to the model as a whole.

**Rewritten, in safe language (observed / measured / directional / decision-support):**

> On a single client-grouped split, the Random Forest's mean absolute error was **measured** at
> 3.2% lower than the frozen Week 4 bucket-average baseline. This is an **observed**, **directional**
> signal that modeling position continuously — rather than in five discrete buckets — may capture
> a real, if modest, improvement; permutation importance attributes nearly all of that gain to
> `avg_position` specifically, with the other four features contributing close to nothing. This is
> not yet a **decision-support**-level claim: it rests on one split, hasn't been tested across
> multiple folds, and (per Part 2 above) its behavior under a time-aware test — the realistic
> deployment scenario — has now also been measured and is reported honestly above, whichever
> direction it moved.


## 5) Self-Check

- **Two paper findings named with real page citations, not just the video's summary?** Yes — both
  verified against the actual PDF text (pages 5, 27, 29, 36), not assumed from the session
  description alone.

- **Methodology questions framed constructively, the way I'd want my own work reviewed?** Yes —
  both ask "where does the evidence actually come from" rather than "this is wrong," and both
  note where the paper's own caveats already partially address the concern (Finding 1) versus
  where the methodology section is silent (Finding 2).

- **My own model re-run under an honest split, with a real before/after?** Yes — time-aware
  (train Feb, test March), a genuinely different and harder test than Week 5's within-March
  grouped-by-client split, reported at its real value regardless of direction.

- **Leakage audit covers all five features, not just the convenient ones?** Yes — including the
  one (`impressions`) with a legitimate but worth-flagging double-use as both feature and part of
  the target's denominator.

- **All claims use safe language?** The 3.2% baseline-beating claim has been rewritten using
  observed/measured/directional/decision-support framing, replacing the earlier flat "beats the
  baseline by 3.2%" statement.

- **What do I still not know?** The time-aware test showed an essentially zero MAE gap (0.00243 vs 0.00243) between within-March and Feb-trained-March-tested — but this is one Feb→March pair, not multiple month-pairs, so I don't yet know if this holds going further back (e.g. Jan→March) or whether it's specific to how stable this particular two-month window happened to be. The paper's own Finding 2 methodology gap (what was actually held out in its 80/20 split) remains genuinely unresolved from the outside; without access to FlyRank's own validation code, the methodology question can be asked but not definitively answered.
